# Chapter 14: SAR Processing

Complete SAR preprocessing pipeline with 3 production bug-fixes.

In [ ]:
import sys
sys.path.insert(0, "../..")
from pygeovision.data.processors.sar import (
    verify_sar_downloads, validate_sar_georeference,
    despeckle_sar, linear_to_db, normalise_sar_for_ai, clip_sar_to_bbox,
)
print("SAR modules loaded")

## Bug-fix verification

In [ ]:
# BUG 1: Identity-transform CRS detection
from pygeovision.data.validators.georeference import validate_georeference
print("BUG 1 fix: validate_georeference() detects identity-transform CRS corruption")

# BUG 2: Download completeness
from pygeovision.data.validators.georeference import check_download_complete
print("BUG 2 fix: check_download_complete() runs tile-read verification")

# BUG 3: CRS-aware bbox clipping
from pygeovision.data.validators.georeference import reproject_bbox_to_raster_crs
print("BUG 3 fix: reproject_bbox_to_raster_crs() auto-converts WGS84 -> raster CRS")

## Correct pipeline order

In [ ]:
print("Correct SAR pipeline order:")
steps = [
    ("S0", "verify_sar_downloads()          — BUG 2: rejects partial downloads"),
    ("S1", "validate_sar_georeference()     — BUG 1: repairs identity-transform CRS"),
    ("S6", "despeckle_sar(LINEAR!)          — MUST be in linear power"),
    ("S7", "linear_to_db()                  — after despeckle"),
    ("S8", "normalise_sar_for_ai()          — scale to [0,1]"),
    ("S9", "clip_sar_to_bbox()              — BUG 3: auto-reprojects WGS84 bbox"),
]
for step, desc in steps:
    print(f"  {step}: {desc}")

## Synthetic SAR demo

In [ ]:
import numpy as np, tempfile, pathlib, rasterio
from rasterio.transform import from_bounds

# Create synthetic SAR scene
def make_sar(path, H=128, W=128):
    data = np.random.gamma(shape=1.0, scale=0.3, size=(1,H,W)).astype("float32")
    data = np.clip(data, 0, 1)
    transform = from_bounds(-0.3, 5.5, -0.05, 5.7, W, H)
    profile = {"driver":"GTiff","dtype":"float32","count":1,"width":W,"height":H,
               "crs":rasterio.crs.CRS.from_epsg(4326),"transform":transform}
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(data)
    return path

tmp = pathlib.Path(tempfile.mkdtemp())
sar_path = make_sar(str(tmp / "S1_synthetic.tif"))
print(f"Synthetic SAR created: {sar_path}")

with rasterio.open(sar_path) as src:
    arr = src.read(1)
    print(f"SAR stats: min={arr.min():.4f} max={arr.max():.4f} mean={arr.mean():.4f}")

## Visualization

In [ ]:
from pygeovision.viz import RasterViewer
rv = RasterViewer(sar_path)
rv.single_band(band=0, colormap="gray").show()
rv.histogram().show()
print("SAR images typically show speckle noise (granular texture)")